In [7]:
import numpy as np
from scipy.integrate import odeint

Acetaminophen/Water

In [10]:
sat_const = 1.5846e-5, -9.0567e-03, 1.3066   # Solubility: second order polynomial, gPCM/gH2O, K, K^

def Csat(T, const=sat_const):
    """The solubility of paracetamol in water (g PCM/g water) as a function of temperature (in K)"""
    A, B, C = const
    return A*T**2 + B*T + C    # kg/m^3

In [16]:
kb_fit = 1.1011e+24                # Nucleation rate constant, m^-3 s^-1 (FIXED)
b_fit  = 6.196                      # Nucelation Order (FIXED)
kg_fit = 2.6868e-04                 # Growth rate constant, m/s (FIXED)
g_fit  = 1.537                      # Growth order (FIXED)
rho_cry_kv_fit = 314.6828       # Volumetric shape factor of prism approximated as rectangular solid # kg/m^3, crystal density of common stable polymorph

# other paramaters
rad_zero = 0                 # initial radius
V = 1                       # m^3, crystallizer/liquid volume
R = 8.31446261815324         # J/mol/K

def abs_sat(C, T):
    """Cdiff = C - Csat(T)"""
    return np.maximum(0, C - Csat(T))

def B(Cdiff, kb, b):
    """Total nucleation rate (0 if undersaturated)."""
    # if Cdiff < 0:
    #     return 0.0
    return kb * Cdiff**b

def growth(Cdiff, kg, g):
    """Crystal growth rate (m/s). 0 if undersaturated or negative."""
    # if Cdiff < 0:
    #     return 0.0
    return kg * Cdiff**g

#------------add dissolution from tutorial model------------------------

diss_const = 1e-06, 0, 1                   # Dissolution: m/s, J/mol, n/a

def rel_sat(C, T):
    '''
    S = rel_sat(C, T)
    '''
    return (C - Csat(T)) / Csat(T)

def dissolution(T, S, const=diss_const):
    kd, Ed, d = const
    return min(kd * np.exp(-Ed / (R * T)) * S * abs(S)**(d-1), 0)

#----------------------------------------------------------------------

def diffeq(y, t, T, r=0.0):
    """
    Moment balance ODEs + solute balance.

    y = [mu0, mu1, mu2, mu3, C, T]
    Assume V = 1
    cool_rate in K/s
    """
    mu0, mu1, mu2, mu3, C = y

    Cdiff = abs_sat(C, T)
    S = rel_sat(C, T)
    N = B(Cdiff, kb_fit, b_fit)
    G = growth(Cdiff, kg_fit, g_fit)
    D = dissolution(T, S)

    dmu0_dt = N
    dmu1_dt = 1 * (G+D) * mu0 + N * r**1
    dmu2_dt = 2 * (G+D) * mu1 + N * r**2
    dmu3_dt = 3 * (G+D) * mu2 + N * r**3
    rho_sol = 1000 # kg/m^3, density of solution
    dC_dt = -rho_cry_kv_fit/rho_sol * (3 * (G+D) * mu2 + N * r**3)  # mass of crystals per m^3 per s

    return [dmu0_dt, dmu1_dt, dmu2_dt, dmu3_dt, dC_dt]

In [17]:
# Initial conditions
mu0_0 = mu1_0 = mu2_0 = mu3_0 = 0.0
mu0_0 = mu1_0 = mu2_0 = mu3_0 = 0
C_0 = 0.025
# C_0 = Csat(T_init)  #starts at saturation
state_init = [mu0_0, mu1_0, mu2_0, mu3_0, C_0]

T_0 = 44.58 + 273.15
t_array = np.linspace(0, 130*60, 1000)

cool_rate = 3.9/3600

def get_next_d50(t_current, T_next, state_current, step=10):
    mu0, mu1, mu2, mu3, C = state_current
    state_next = odeint(diffeq, state_current, [t_current, t_current + step], args=(T_next,))
    mu0_next, mu1_next, mu2_next, mu3_next, C_next = state_next[-1]
    d50_next = mu1_next / (mu0_next + 1e-12) * 1e6
    return d50_next, state_next[-1]

get_next_d50(0, 44.58 + 273.15 - 3.9/3600 * 10, state_init)

(np.float64(0.0), array([0.   , 0.   , 0.   , 0.   , 0.025]))